In [ ]:
!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex

In [ ]:
!pip install "paddlepaddle==3.2.2"
!pip install "paddleocr[all]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    lang='en'
)

image_folder = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'

ocr_data = []
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')

for filename in os.listdir(image_folder):
    if filename.lower().endswith(valid_extensions):
        img_path = os.path.join(image_folder, filename)

        try:
            result = ocr.predict(img_path)
            text_lines = []
            if result:
                for res in result:
                    if hasattr(res, 'rec_texts'):
                        text_lines.extend(res.rec_texts)
                    elif isinstance(res, dict) and 'rec_texts' in res:
                        text_lines.extend(res['rec_texts'])
                    else:
                        try:
                            res_dict = dict(res)
                            if 'rec_texts' in res_dict:
                                text_lines.extend(res_dict['rec_texts'])
                        except Exception:
                            pass

            extracted_text = "\n".join([str(t) for t in text_lines if t]).strip()

            ocr_data.append({
                "filename": filename,
                "extracted_text": extracted_text
            })

            if extracted_text:
                print(f"Successfully processed {filename}: Found {len(text_lines)} text blocks.")
            else:
                print(f"Processed {filename}, but no text was detected.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

os.makedirs(os.path.dirname(output_json), exist_ok=True)

with open(output_json, 'w', encoding='utf-8') as json_file:
    json.dump(ocr_data, json_file, ensure_ascii=False, indent=4)

print(f"\nAll done! Results saved to: {output_json}")